# Input validation and sanitization  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.5.a: Field validator (allowlist + injection detector)

**Skill:** Mitigate Risk · **Maps to:** 5.5.A · **Language:** Python 3 (pytest)

## Problem
Build the server-side validation gate described in the lesson: given a batch of submitted form
field values, classify each as SAFE or BLOCKED using **allowlist rules per field type** plus
detection of known **injection meta-characters/keywords**, the same two checks a real
input-validation layer runs before any value reaches a database query, HTML page, or shell.

## Function contract
Implement `solve(submissions: list) -> list` in the code editor (see the Reference section below).

## Input
`submissions`: `list[dict]`, each `{"field": str, "type": str, "value": str}`. `type` is one of
`"zip"`, `"email"`, `"quantity"`, `"username"`.

## Output
`list[str]` of the `value`s that are BLOCKED, in the same order they appeared in the input.

## Allowlist rules (per `type`)
- `zip`: exactly 5 digits.
- `email`: `local@domain.tld` shape.
- `quantity`: integer string `1`-`99`.
- `username`: 3-20 letters/digits/underscore.

A value is also BLOCKED (even if it matches the allowlist shape) if it contains a known
injection signature: `--`, `;`, `<script`, `UNION SELECT`, `OR 1=1`, `' OR '`, `DROP TABLE`,
`../`, `%3B`, `%2F` (case-insensitive).

## Examples
```
[{"field":"zip","type":"zip","value":"94107"}]                    ->  []
[{"field":"zip","type":"zip","value":"9410"}]                     ->  ["9410"]
[{"field":"u","type":"username","value":"admin'--"}]               ->  ["admin'--"]
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(submissions: list) -> list
```

### Input (`submissions`)
`list[dict]`, each `{"field": str, "type": str, "value": str}`. `type` is one of `"zip"`,
`"email"`, `"quantity"`, `"username"`.

### Output (`result`)
`list[str]`, the `value` of every submission that is **BLOCKED**, in the same order as the
input. A submission is blocked if either condition holds:
1. Its `value` fails the allowlist pattern for its `type` (server-side shape check).
2. Its `value` matches a known injection meta-character/keyword signature (checked
   independently of `type`, a value can match the type's shape and still be blocked).

### Allowlist rules
- `zip`: exactly 5 digits.
- `email`: `local@domain.tld` shape (letters/digits/`._+-` local part, dotted domain).
- `quantity`: integer string from `1` to `99` inclusive (no leading zero, no `0`, no `100+`).
- `username`: 3-20 characters, letters/digits/underscore only.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Injection signatures include (case-insensitive): `--`, `;`, `<script`, `UNION SELECT`,
  `OR 1=1`, `' OR '`, `DROP TABLE`, `../` path traversal, and URL-encoded `%3B`/`%2F`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.5.a. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
import re
from typing import Dict, List

# Allowlist patterns per field type (server-side rules — see atom.md).
_ALLOW = {
    "zip": re.compile(r"^\d{5}$"),
    "email": re.compile(r"^[A-Za-z0-9._+-]+@[A-Za-z0-9-]+\.[A-Za-z0-9.-]+$"),
    "quantity": re.compile(r"^(?:[1-9][0-9]?)$"),  # 1-99
    "username": re.compile(r"^[A-Za-z0-9_]{3,20}$"),
}

# Known injection meta-character / keyword signatures, checked regardless of type.
_INJECTION = re.compile(
    r"(--|;|<script|</script|UNION\s+SELECT|OR\s+1\s*=\s*1|'\s*OR\s*'|\bDROP\s+TABLE\b|"
    r"\.\./|%3B|%2F)",
    re.IGNORECASE,
)


def solve(submissions: List[Dict[str, str]]) -> List[str]:
    """Classify each submission; return the `value`s that are BLOCKED, in input order.

    A submission is BLOCKED if its value fails the type's allowlist pattern, or if it
    contains a known injection meta-character/keyword signature (checked independent of type).
    """
    blocked = []
    for sub in submissions:
        field_type = sub["type"]
        value = sub["value"]
        pattern = _ALLOW.get(field_type)
        allow_ok = bool(pattern and pattern.match(value))
        has_injection = bool(_INJECTION.search(value))
        if not allow_ok or has_injection:
            blocked.append(value)
    return blocked


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_all_valid_none_blocked():
    subs = [
        {"field": "zip", "type": "zip", "value": "94107"},
        {"field": "email", "type": "email", "value": "user@example.com"},
        {"field": "qty", "type": "quantity", "value": "5"},
    ]
    assert impl.solve(subs) == []


def test_bad_zip_blocked():
    subs = [{"field": "zip", "type": "zip", "value": "9410"}]
    assert impl.solve(subs) == ["9410"]


def test_sql_injection_blocked_even_if_type_matches_shape():
    subs = [{"field": "username", "type": "username", "value": "admin'--"}]
    assert impl.solve(subs) == ["admin'--"]


def test_xss_payload_blocked():
    subs = [{"field": "username", "type": "username", "value": "<script>x</script>"}]
    assert impl.solve(subs) == ["<script>x</script>"]


def test_order_preserved_and_only_blocked_returned():
    subs = [
        {"field": "zip", "type": "zip", "value": "94107"},
        {"field": "zip", "type": "zip", "value": "abcde"},
        {"field": "qty", "type": "quantity", "value": "150"},
        {"field": "email", "type": "email", "value": "a@b.com"},
    ]
    assert impl.solve(subs) == ["abcde", "150"]


# ---- hidden / edge cases -------------------------------------------------
def test_quantity_zero_and_out_of_range_blocked():
    subs = [
        {"field": "qty", "type": "quantity", "value": "0"},
        {"field": "qty", "type": "quantity", "value": "100"},
        {"field": "qty", "type": "quantity", "value": "99"},
    ]
    assert impl.solve(subs) == ["0", "100"]


def test_union_select_blocked_regardless_of_case():
    subs = [{"field": "search", "type": "username", "value": "foo union select 1"}]
    assert impl.solve(subs) == ["foo union select 1"]


def test_path_traversal_blocked():
    subs = [{"field": "file", "type": "username", "value": "../etc/passwd"}]
    assert impl.solve(subs) == ["../etc/passwd"]


def test_empty_input():
    assert impl.solve([]) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
